## System Architecture

- **AMAN**: arrival sequencing
- **DMAN**: departure sequencing + ATFM constraints
- **GENERATOR**: adversarial scenario mutation
- **SUPERVISOR**: rotating preference profile

Training signal is role-specific reward shaping with cross-role conflict penalties and supervisor alignment.

In [ ]:
import os
import sys
from pathlib import Path

# Offline + deterministic constraints
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('REWARD_FAILURE_MODE', 'strict')

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'training').exists():
            return p
        if (p / 'tasks.py').exists() and (p / 'training').exists():
            return p
    return cur.resolve()

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workspace root:', ROOT)
print('TRANSFORMERS_OFFLINE=', os.getenv('TRANSFORMERS_OFFLINE'))
print('HF_HUB_OFFLINE=', os.getenv('HF_HUB_OFFLINE'))
print('training module path exists:', (ROOT / 'training').exists())

In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Model Loading (offline, cache-only)

We resolve model files from local cache only and avoid any live Hub lookup.

In [ ]:
MODEL_REF = 'Qwen/Qwen2.5-7B-Instruct'  # or local path

def resolve_model_path(model_ref: str) -> str:
    p = Path(model_ref)
    if p.exists():
        return str(p)
    return snapshot_download(repo_id=model_ref, local_files_only=True, resume_download=True)

model_path = resolve_model_path(MODEL_REF)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

print('Resolved model path:', model_path)
print('dtype:', dtype)

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    torch_dtype=dtype,
    device_map='auto' if torch.cuda.is_available() else None,
)
print('Model loaded.')

## LoRA Setup

LoRA is used to keep optimization lightweight and stable on constrained server setup.
We avoid full-model finetuning for memory and recovery safety.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print('LoRA attached and optimizer created.')

## Dataset / Episode Simulation

Each sample is one role-turn containing chat prompt + role metadata.

In [ ]:
# Self-healing import bootstrap (works even if setup cell was not run)
import os
import sys
from pathlib import Path
from typing import Optional

def _find_repo_root(start: Path) -> Optional[Path]:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'training').exists() and ((p / 'pyproject.toml').exists() or (p / 'tasks.py').exists()):
            return p
    return None

def _pick_repo_root() -> Path:
    # 1) explicit override
    _env_root = os.getenv('ATC_REPO_ROOT', '').strip()
    if _env_root and Path(_env_root).exists():
        return Path(_env_root).resolve()

    # 2) discover from cwd
    _cwd_root = _find_repo_root(Path.cwd())
    if _cwd_root is not None:
        return _cwd_root

    # 3) common fallback locations
    for candidate in [Path('/home/keshav/ats'), Path.home() / 'ats', Path.home() / 'workspace' / 'ats']:
        if candidate.exists() and (candidate / 'training').exists():
            return candidate.resolve()

    raise RuntimeError(
        "Could not locate repo root with 'training/' package. "
        "Set ATC_REPO_ROOT to your repository path before running this cell."
    )

_root = _pick_repo_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from training.dataset import build_episode_dataset

samples = build_episode_dataset(n_episodes=5, seed=42, include_generator=True, include_supervisor=True)
print('Repo root used:', _root)
print('Python cwd:', Path.cwd())
print('Total samples:', len(samples))
print('Sample keys:', list(samples[0].keys()))
print('Sample role:', samples[0]['agent_role'])

## Reward Function Overview

Reward logic is reused from repository code (unchanged scoring behavior):
- `aman_reward_fn`: delay, emergency handling, coverage, ToM bonus, supervisor alignment, normalized conflict penalty
- `dman_reward_fn`: delay, ATFM compliance, emergency handling, coverage, ToM bonus, supervisor alignment, normalized conflict penalty
- `generator_reward_fn`: adversarial reward with solvability guard
- `supervisor_reward_fn`: preference-alignment scoring and calibration

This notebook does not alter those formulas.

In [ ]:
# Self-healing import bootstrap for reward modules
import os
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'training').exists() and ((p / 'pyproject.toml').exists() or (p / 'tasks.py').exists()):
            return p
    return start.resolve()

_env_root = os.getenv('ATC_REPO_ROOT', '').strip()
if _env_root and Path(_env_root).exists():
    _root = Path(_env_root).resolve()
else:
    _root = _find_repo_root(Path.cwd())

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from training.reward_functions import aman_reward_fn, dman_reward_fn, generator_reward_fn, supervisor_reward_fn

ROLE_TO_REWARD_FN = {
    'AMAN': aman_reward_fn,
    'DMAN': dman_reward_fn,
    'GENERATOR': generator_reward_fn,
    'SUPERVISOR': supervisor_reward_fn,
}

def compute_reward(sample, completion):
    role = sample['agent_role']
    fn = ROLE_TO_REWARD_FN[role]

    if role == 'AMAN':
        return fn([completion],
                  task_id=[sample['task_id']],
                  supervisor_profile=[sample['supervisor_profile']],
                  dman_slots_json=[sample.get('dman_slots_json', '[]')],
                  atfm_deadlines_json=[sample.get('atfm_deadlines_json', '{}')])[0]

    if role == 'DMAN':
        return fn([completion],
                  task_id=[sample['task_id']],
                  supervisor_profile=[sample['supervisor_profile']],
                  aman_slots_json=[sample.get('aman_slots_json', '[]')],
                  atfm_deadlines_json=[sample.get('atfm_deadlines_json', '{}')])[0]

    if role == 'GENERATOR':
        return fn([completion],
                  task_id=[sample['task_id']],
                  controller_scores=[float(sample.get('controller_scores', 0.5))])[0]

    return fn([completion],
              task_id=[sample['task_id']],
              supervisor_profile=[sample['supervisor_profile']],
              merged_plan_json=[sample.get('merged_plan_json', '[]')])[0]

print('Reward modules imported from root:', _root)

## Training Loop (Harness)

This is a controlled offline harness loop for runtime validation.
It logs role, reward, and loss per step.

In [ ]:
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from multi_agent.environment import MultiAgentATCEnvironment
from multi_agent.generator import ChallengeGenerator
from multi_agent.supervisor import SupervisorAgent
from multi_agent.models import SUPERVISOR_PROFILES
from multi_agent.inference import _build_aman_heuristic, _build_dman_heuristic
from training.dataset import AMAN_SYSTEM, DMAN_SYSTEM, parse_aman_action, parse_dman_action
from tasks import ordered_tasks, task_catalog

# Keep server-safe constraints/workarounds consistent
MAX_EPISODES = 12
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 128
NEGOTIATE_ROUNDS = 1
USE_GENERATOR = True
GENERATOR_WARMUP_EPISODES = 2  # stabilize before adversarial curriculum
SEED = 42
LIVE_PLOT_EVERY = 1  # update every training query
ROLLING_WINDOW = 5

task_ids = [t.task_id for t in ordered_tasks()]
catalog = task_catalog()

env = MultiAgentATCEnvironment(seed=SEED)
generator = ChallengeGenerator(seed=SEED)
supervisor = SupervisorAgent()

query_logs = []
episode_logs = []
global_step = 0

def _rolling_mean(values, window=ROLLING_WINDOW):
    s = pd.Series(values, dtype='float64')
    return s.rolling(window=min(window, max(1, len(s))), min_periods=1).mean().to_numpy()

def _safe_legend(ax, **kwargs):
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(**kwargs)

def create_live_panel():
    plt.ion()
    fig, axes = plt.subplots(3, 3, figsize=(19, 13))
    return fig, axes

def refresh_training_panel(query_logs, episode_logs, fig, axes, force=False):
    if len(query_logs) == 0:
        return
    if (not force) and (len(query_logs) % LIVE_PLOT_EVERY != 0):
        return

    qdf = pd.DataFrame(query_logs)
    edf = pd.DataFrame(episode_logs) if episode_logs else pd.DataFrame()

    for ax in axes.flat:
        ax.clear()

    # 1) Reward over global steps + rolling trend
    ax = axes[0, 0]
    for role, grp in qdf.groupby('role'):
        grp = grp.sort_values('step')
        ax.plot(grp['step'], grp['reward'], alpha=0.30, marker='.', linewidth=1.0, label=f'{role} raw')
        ax.plot(grp['step'], _rolling_mean(grp['reward']), linewidth=2.0, label=f'{role} roll')
    ax.set_title('Reward by Global Step (raw + rolling)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Reward')
    ax.grid(True)
    _safe_legend(ax, fontsize=7, ncol=2)

    # 2) Weighted loss over global steps + rolling
    ax = axes[0, 1]
    for role, grp in qdf.groupby('role'):
        grp = grp.sort_values('step')
        y = pd.Series(grp['weighted_loss']).replace([np.inf, -np.inf], np.nan).fillna(method='ffill').fillna(method='bfill').fillna(0)
        ax.plot(grp['step'], y, alpha=0.30, marker='.', linewidth=1.0, label=f'{role} raw')
        ax.plot(grp['step'], _rolling_mean(y), linewidth=2.0, label=f'{role} roll')
    ax.set_title('Weighted Loss by Step (raw + rolling)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Weighted Loss')
    ax.grid(True)
    _safe_legend(ax, fontsize=7, ncol=2)

    # 3) Parse success cumulative mean by role
    ax = axes[0, 2]
    for role, grp in qdf.groupby('role'):
        grp = grp.sort_values('step')
        cumulative = grp['parse_ok'].astype(float).expanding().mean()
        ax.plot(grp['step'], cumulative, marker='o', linewidth=1.8, label=role)
    ax.set_title('Parse Success Cumulative Mean')
    ax.set_xlabel('Step')
    ax.set_ylabel('Parse Success')
    ax.set_ylim(0, 1.05)
    ax.grid(True)
    _safe_legend(ax, fontsize=8)

    # 4) Episode macro metrics (composite / coord / conflicts)
    ax = axes[1, 0]
    if len(edf):
        edf = edf.sort_values('episode')
        ax.plot(edf['episode'], edf['composite_score'], marker='o', label='composite')
        ax.plot(edf['episode'], edf['coord_score'], marker='o', label='coord')
        ax.plot(edf['episode'], edf['conflicts'], marker='x', label='conflicts')
    ax.set_title('Episode Macro Metrics')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Value')
    ax.grid(True)
    _safe_legend(ax, fontsize=8)

    # 5) Episode role rewards
    ax = axes[1, 1]
    if len(edf):
        ax.plot(edf['episode'], edf['aman_reward'], marker='o', label='AMAN reward')
        ax.plot(edf['episode'], edf['dman_reward'], marker='o', label='DMAN reward')
        ax.plot(edf['episode'], edf['generator_reward'], marker='x', label='Generator reward')
        ax.plot(edf['episode'], edf['supervisor_score'], marker='x', label='Supervisor score')
    ax.set_title('Per-Episode Role Rewards')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward / Score')
    ax.grid(True)
    _safe_legend(ax, fontsize=7, ncol=2)

    # 6) Reward distribution by role
    ax = axes[1, 2]
    bins = np.linspace(-1.0, 1.0, 16)
    for role, grp in qdf.groupby('role'):
        ax.hist(grp['reward'], bins=bins, alpha=0.35, label=role)
    ax.set_title('Reward Distribution by Role')
    ax.set_xlabel('Reward')
    ax.set_ylabel('Count')
    ax.grid(True)
    _safe_legend(ax, fontsize=8)

    # 7) Prompt/Completion size trends
    ax = axes[2, 0]
    for role, grp in qdf.groupby('role'):
        grp = grp.sort_values('step')
        ax.plot(grp['step'], _rolling_mean(grp['prompt_chars']), linewidth=2.0, label=f'{role} prompt')
        ax.plot(grp['step'], _rolling_mean(grp['completion_chars']), linestyle='--', linewidth=1.8, label=f'{role} completion')
    ax.set_title('Prompt vs Completion Length (rolling)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Chars')
    ax.grid(True)
    _safe_legend(ax, fontsize=7, ncol=2)

    # 8) Mean composite by task
    ax = axes[2, 1]
    if len(edf):
        mean_by_task = edf.groupby('task_id')['composite_score'].mean().sort_values()
        ax.bar(mean_by_task.index.astype(str), mean_by_task.values, color='tab:green', alpha=0.8)
        ax.tick_params(axis='x', labelrotation=20)
    ax.set_title('Average Composite Score by Task')
    ax.set_xlabel('Task')
    ax.set_ylabel('Avg Composite')
    ax.grid(True, axis='y')

    # 9) Mean composite by supervisor profile
    ax = axes[2, 2]
    if len(edf):
        mean_by_sup = edf.groupby('supervisor_profile')['composite_score'].mean().sort_values()
        ax.bar(mean_by_sup.index.astype(str), mean_by_sup.values, color='tab:orange', alpha=0.8)
        ax.tick_params(axis='x', labelrotation=20)
    ax.set_title('Average Composite by Supervisor Profile')
    ax.set_xlabel('Supervisor Profile')
    ax.set_ylabel('Avg Composite')
    ax.grid(True, axis='y')

    fig.tight_layout()
    clear_output(wait=True)
    display(fig)

def generate_completion(system_text: str, user_text: str) -> str:
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def train_one_query(system_text: str, user_text: str, completion: str, reward: float):
    text = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}],
        tokenize=False,
        add_generation_prompt=True,
    ) + completion
    batch = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    batch = {k: v.to(model.device) for k, v in batch.items()}

    model.train()
    outputs = model(**batch, labels=batch['input_ids'])
    ce_loss = outputs.loss

    reward = max(-1.0, min(1.0, float(reward)))
    weight = max(0.1, 1.0 - reward)
    weighted_loss = ce_loss * weight

    if torch.isnan(weighted_loss) or torch.isinf(weighted_loss):
        optimizer.zero_grad(set_to_none=True)
        return float(ce_loss.item()), float('nan')

    weighted_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    return float(ce_loss.item()), float(weighted_loss.item())

live_fig, live_axes = create_live_panel()

for ep in range(MAX_EPISODES):
    task_id = task_ids[ep % len(task_ids)]
    profile = supervisor.sample_profile(ep)
    sup_desc = SUPERVISOR_PROFILES[profile]['description']

    base_task = catalog[task_id]
    generator_active = USE_GENERATOR and ep >= GENERATOR_WARMUP_EPISODES
    if generator_active:
        task_for_episode, _ = generator.mutate(base_task)
    else:
        task_for_episode = base_task

    aman_obs, dman_obs = env.reset(
        task_id=task_id,
        episode_id=ep,
        supervisor_profile=profile,
        mutated_task=task_for_episode,
    )
    atfm = env._state.atfm_deadlines

    aman_system = AMAN_SYSTEM + f"\n\nSUPERVISOR TODAY: {sup_desc}"
    dman_system = DMAN_SYSTEM + f"\n\nSUPERVISOR TODAY: {sup_desc}"

    model.eval()
    aman_completion_bid = generate_completion(aman_system, aman_obs.to_prompt_text())
    dman_completion_bid = generate_completion(dman_system, dman_obs.to_prompt_text())

    aman_action_bid = parse_aman_action(aman_completion_bid)
    dman_action_bid = parse_dman_action(dman_completion_bid)

    aman_parse_ok_bid = aman_action_bid is not None
    dman_parse_ok_bid = dman_action_bid is not None

    if not aman_parse_ok_bid:
        aman_action_bid = _build_aman_heuristic(aman_obs)
    if not dman_parse_ok_bid:
        dman_action_bid = _build_dman_heuristic(dman_obs, atfm)

    aman_obs_2, dman_obs_2, _, done = env.step_bid(aman_action_bid, dman_action_bid)

    aman_completion_neg = ''
    dman_completion_neg = ''
    aman_parse_ok_neg = None
    dman_parse_ok_neg = None

    if (not done) and NEGOTIATE_ROUNDS > 0:
        aman_completion_neg = generate_completion(aman_system, aman_obs_2.to_prompt_text())
        dman_completion_neg = generate_completion(dman_system, dman_obs_2.to_prompt_text())

        aman_action_neg = parse_aman_action(aman_completion_neg)
        dman_action_neg = parse_dman_action(dman_completion_neg)
        aman_parse_ok_neg = aman_action_neg is not None
        dman_parse_ok_neg = dman_action_neg is not None

        if not aman_parse_ok_neg:
            aman_action_neg = _build_aman_heuristic(aman_obs_2)
        if not dman_parse_ok_neg:
            dman_action_neg = _build_dman_heuristic(dman_obs_2, atfm)

        env.step_negotiate(aman_action_neg, dman_action_neg)

    result = env.finalize()
    if generator_active:
        generator.update(result.composite_score)

    aman_ce_bid, aman_w_bid = train_one_query(
        aman_system, aman_obs.to_prompt_text(), aman_completion_bid, result.aman_reward
    )
    query_logs.append({
        'step': global_step,
        'episode': ep,
        'task_id': task_id,
        'role': 'AMAN',
        'round': 'BID',
        'parse_ok': aman_parse_ok_bid,
        'reward': float(result.aman_reward),
        'ce_loss': aman_ce_bid,
        'weighted_loss': aman_w_bid,
        'prompt_chars': len(aman_obs.to_prompt_text()),
        'completion_chars': len(aman_completion_bid),
        'composite_score': float(result.composite_score),
        'coord_score': float(result.per_role.coordination_score),
        'conflicts': int(result.per_role.cross_lane_conflicts),
        'supervisor_profile': profile.value,
    })
    global_step += 1
    refresh_training_panel(query_logs, episode_logs, live_fig, live_axes)

    dman_ce_bid, dman_w_bid = train_one_query(
        dman_system, dman_obs.to_prompt_text(), dman_completion_bid, result.dman_reward
    )
    query_logs.append({
        'step': global_step,
        'episode': ep,
        'task_id': task_id,
        'role': 'DMAN',
        'round': 'BID',
        'parse_ok': dman_parse_ok_bid,
        'reward': float(result.dman_reward),
        'ce_loss': dman_ce_bid,
        'weighted_loss': dman_w_bid,
        'prompt_chars': len(dman_obs.to_prompt_text()),
        'completion_chars': len(dman_completion_bid),
        'composite_score': float(result.composite_score),
        'coord_score': float(result.per_role.coordination_score),
        'conflicts': int(result.per_role.cross_lane_conflicts),
        'supervisor_profile': profile.value,
    })
    global_step += 1
    refresh_training_panel(query_logs, episode_logs, live_fig, live_axes)

    if (not done) and NEGOTIATE_ROUNDS > 0:
        aman_ce_neg, aman_w_neg = train_one_query(
            aman_system, aman_obs_2.to_prompt_text(), aman_completion_neg, result.aman_reward
        )
        query_logs.append({
            'step': global_step,
            'episode': ep,
            'task_id': task_id,
            'role': 'AMAN',
            'round': 'NEGOTIATE',
            'parse_ok': bool(aman_parse_ok_neg),
            'reward': float(result.aman_reward),
            'ce_loss': aman_ce_neg,
            'weighted_loss': aman_w_neg,
            'prompt_chars': len(aman_obs_2.to_prompt_text()),
            'completion_chars': len(aman_completion_neg),
            'composite_score': float(result.composite_score),
            'coord_score': float(result.per_role.coordination_score),
            'conflicts': int(result.per_role.cross_lane_conflicts),
            'supervisor_profile': profile.value,
        })
        global_step += 1
        refresh_training_panel(query_logs, episode_logs, live_fig, live_axes)

        dman_ce_neg, dman_w_neg = train_one_query(
            dman_system, dman_obs_2.to_prompt_text(), dman_completion_neg, result.dman_reward
        )
        query_logs.append({
            'step': global_step,
            'episode': ep,
            'task_id': task_id,
            'role': 'DMAN',
            'round': 'NEGOTIATE',
            'parse_ok': bool(dman_parse_ok_neg),
            'reward': float(result.dman_reward),
            'ce_loss': dman_ce_neg,
            'weighted_loss': dman_w_neg,
            'prompt_chars': len(dman_obs_2.to_prompt_text()),
            'completion_chars': len(dman_completion_neg),
            'composite_score': float(result.composite_score),
            'coord_score': float(result.per_role.coordination_score),
            'conflicts': int(result.per_role.cross_lane_conflicts),
            'supervisor_profile': profile.value,
        })
        global_step += 1
        refresh_training_panel(query_logs, episode_logs, live_fig, live_axes)

    episode_logs.append({
        'episode': ep,
        'task_id': task_id,
        'supervisor_profile': profile.value,
        'composite_score': float(result.composite_score),
        'coord_score': float(result.per_role.coordination_score),
        'conflicts': int(result.per_role.cross_lane_conflicts),
        'aman_reward': float(result.aman_reward),
        'dman_reward': float(result.dman_reward),
        'generator_reward': float(result.generator_reward),
        'supervisor_score': float(result.supervisor_score),
        'neg_rounds': int(result.negotiation_rounds),
    })
    refresh_training_panel(query_logs, episode_logs, live_fig, live_axes, force=True)

    print(
        f"ep={ep:02d} task={task_id} composite={result.composite_score:.3f} "
        f"coord={result.per_role.coordination_score:.3f} conflicts={result.per_role.cross_lane_conflicts} "
        f"AMAN={result.aman_reward:+.3f} DMAN={result.dman_reward:+.3f} generator_on={generator_active}"
    )

refresh_training_panel(query_logs, episode_logs, live_fig, live_axes, force=True)
plt.ioff()
print('Training run complete.')
print(f'episodes={len(episode_logs)} queries={len(query_logs)} tasks_used={sorted(set([e["task_id"] for e in episode_logs]))}')

## Visualizations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

query_df = pd.DataFrame(query_logs)
episode_df = pd.DataFrame(episode_logs)

print('query_df shape:', query_df.shape)
print('episode_df shape:', episode_df.shape)
display(episode_df.head(3))
display(query_df.head(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Query-level reward over global steps (role split)
for role, grp in query_df.groupby('role'):
    axes[0, 0].plot(grp['step'], grp['reward'], marker='o', linewidth=1.5, label=role)
axes[0, 0].set_title('Query Reward over Global Steps (Role Split)')
axes[0, 0].set_xlabel('Global Step')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].grid(True)
axes[0, 0].legend()

# 2) Query-level CE loss over steps
for role, grp in query_df.groupby('role'):
    axes[0, 1].plot(grp['step'], grp['ce_loss'], marker='o', linewidth=1.5, label=role)
axes[0, 1].set_title('Query CE Loss over Global Steps (Role Split)')
axes[0, 1].set_xlabel('Global Step')
axes[0, 1].set_ylabel('CE Loss')
axes[0, 1].grid(True)
axes[0, 1].legend()

# 3) Episode composite by task
for task_id, grp in episode_df.groupby('task_id'):
    axes[1, 0].plot(grp['episode'], grp['composite_score'], marker='o', linewidth=1.5, label=task_id)
axes[1, 0].set_title('Episode Composite Score by Task')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Composite Score')
axes[1, 0].grid(True)
axes[1, 0].legend(fontsize=8)

# 4) Episode coordination and conflicts
axes[1, 1].plot(episode_df['episode'], episode_df['coord_score'], marker='o', label='coord_score')
axes[1, 1].plot(episode_df['episode'], episode_df['conflicts'], marker='x', label='conflicts')
axes[1, 1].set_title('Coordination vs Conflicts per Episode')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Value')
axes[1, 1].grid(True)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Avg reward by task and role (grouped bars)
pivot_task_role = query_df.pivot_table(index='task_id', columns='role', values='reward', aggfunc='mean').fillna(0.0)
pivot_task_role.plot(kind='bar', ax=axes[0, 0])
axes[0, 0].set_title('Average Reward by Task × Role')
axes[0, 0].set_xlabel('Task')
axes[0, 0].set_ylabel('Avg Reward')
axes[0, 0].grid(True, axis='y')

# 2) Parse success rate by role and round
parse_rate = query_df.groupby(['role', 'round'])['parse_ok'].mean().unstack(fill_value=0.0)
parse_rate.plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Parse Success Rate by Role × Round')
axes[0, 1].set_xlabel('Role')
axes[0, 1].set_ylabel('Parse Success Rate')
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].grid(True, axis='y')

# 3) Prompt/completion lengths by role
for role, grp in query_df.groupby('role'):
    axes[1, 0].scatter(grp['prompt_chars'], grp['completion_chars'], alpha=0.7, label=role)
axes[1, 0].set_title('Prompt vs Completion Length (Role Split)')
axes[1, 0].set_xlabel('Prompt Chars')
axes[1, 0].set_ylabel('Completion Chars')
axes[1, 0].grid(True)
axes[1, 0].legend()

# 4) Reward vs weighted loss by role
for role, grp in query_df.groupby('role'):
    y = grp['weighted_loss'].replace([np.inf, -np.inf], np.nan)
    axes[1, 1].scatter(grp['reward'], y, alpha=0.7, label=role)
axes[1, 1].set_title('Reward vs Weighted Loss (Role Split)')
axes[1, 1].set_xlabel('Reward')
axes[1, 1].set_ylabel('Weighted Loss')
axes[1, 1].grid(True)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Heatmap: task x role reward
heat_reward = query_df.pivot_table(index='task_id', columns='role', values='reward', aggfunc='mean').fillna(0.0)
if HAS_SEABORN:
    sns.heatmap(heat_reward, annot=True, fmt='.3f', cmap='coolwarm', ax=axes[0, 0])
else:
    im = axes[0, 0].imshow(heat_reward.values, aspect='auto', cmap='coolwarm')
    axes[0, 0].set_xticks(range(len(heat_reward.columns)))
    axes[0, 0].set_xticklabels(heat_reward.columns, rotation=45, ha='right')
    axes[0, 0].set_yticks(range(len(heat_reward.index)))
    axes[0, 0].set_yticklabels(heat_reward.index)
    fig.colorbar(im, ax=axes[0, 0])
axes[0, 0].set_title('Heatmap: Avg Reward (Task × Role)')

# 2) Heatmap: task x role ce_loss
heat_loss = query_df.pivot_table(index='task_id', columns='role', values='ce_loss', aggfunc='mean').fillna(0.0)
if HAS_SEABORN:
    sns.heatmap(heat_loss, annot=True, fmt='.3f', cmap='viridis', ax=axes[0, 1])
else:
    im2 = axes[0, 1].imshow(heat_loss.values, aspect='auto', cmap='viridis')
    axes[0, 1].set_xticks(range(len(heat_loss.columns)))
    axes[0, 1].set_xticklabels(heat_loss.columns, rotation=45, ha='right')
    axes[0, 1].set_yticks(range(len(heat_loss.index)))
    axes[0, 1].set_yticklabels(heat_loss.index)
    fig.colorbar(im2, ax=axes[0, 1])
axes[0, 1].set_title('Heatmap: Avg CE Loss (Task × Role)')

# 3) Distribution: query rewards
if HAS_SEABORN:
    sns.violinplot(data=query_df, x='role', y='reward', inner='quartile', ax=axes[1, 0])
else:
    query_df.boxplot(column='reward', by='role', ax=axes[1, 0])
axes[1, 0].set_title('Reward Distribution by Role')
axes[1, 0].set_xlabel('Role')
axes[1, 0].set_ylabel('Reward')
axes[1, 0].grid(True, axis='y')

# 4) Distribution: per-task composite
if HAS_SEABORN:
    sns.boxplot(data=episode_df, x='task_id', y='composite_score', ax=axes[1, 1])
else:
    episode_df.boxplot(column='composite_score', by='task_id', ax=axes[1, 1], rot=45)
axes[1, 1].set_title('Composite Score Distribution by Task')
axes[1, 1].set_xlabel('Task')
axes[1, 1].set_ylabel('Composite Score')
axes[1, 1].grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Per-task query count split by role
count_table = query_df.groupby(['task_id', 'role']).size().unstack(fill_value=0)
count_table.plot(kind='bar', stacked=True, ax=axes[0, 0])
axes[0, 0].set_title('Query Count by Task × Role')
axes[0, 0].set_xlabel('Task')
axes[0, 0].set_ylabel('Query Count')
axes[0, 0].grid(True, axis='y')

# 2) Reward moving average per role
for role, grp in query_df.sort_values('step').groupby('role'):
    rolling = grp['reward'].rolling(window=max(2, min(5, len(grp))), min_periods=1).mean()
    axes[0, 1].plot(grp['step'], rolling, linewidth=2, label=role)
axes[0, 1].set_title('Reward Moving Average by Role')
axes[0, 1].set_xlabel('Global Step')
axes[0, 1].set_ylabel('Rolling Mean Reward')
axes[0, 1].grid(True)
axes[0, 1].legend()

# 3) Parse success by task
parse_by_task = query_df.groupby('task_id')['parse_ok'].mean().sort_values()
parse_by_task.plot(kind='bar', ax=axes[1, 0], color='tab:purple')
axes[1, 0].set_title('Parse Success Rate by Task')
axes[1, 0].set_xlabel('Task')
axes[1, 0].set_ylabel('Parse Success Rate')
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].grid(True, axis='y')

# 4) Episode-level reward split
axes[1, 1].plot(episode_df['episode'], episode_df['aman_reward'], marker='o', label='aman_reward')
axes[1, 1].plot(episode_df['episode'], episode_df['dman_reward'], marker='o', label='dman_reward')
axes[1, 1].plot(episode_df['episode'], episode_df['generator_reward'], marker='o', label='generator_reward')
axes[1, 1].plot(episode_df['episode'], episode_df['supervisor_score'], marker='o', label='supervisor_score')
axes[1, 1].set_title('Episode-Level Reward/Score Split')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Value')
axes[1, 1].grid(True)
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('--- Aggregated Tables ---')
display(query_df.groupby(['task_id', 'role'])[['reward', 'ce_loss', 'weighted_loss', 'parse_ok']].mean().round(4))
display(episode_df.groupby('task_id')[['composite_score', 'coord_score', 'conflicts', 'aman_reward', 'dman_reward']].mean().round(4))

## Observations

Fill this section after execution with concrete outcomes, e.g.:
- supervisor role reward trend
- generator penalty behavior
- AMAN/DMAN stability or oscillation
- whether loss decreases with bounded rewards

## Limitations

- This harness is not full GRPO trajectory/group optimization.
- Reward scaling here is a stability proxy, not final policy objective.
- No multi-sample policy grouping or KL-controlled GRPO updates in this notebook run.

## Next Steps

1. Keep this harness as server-stability gate.
2. Move to full GRPO runner after environment lock is proven.
3. Add per-role moving averages and longer-run diagnostics once stable.